In [2]:
import pandas as pd
import matplotlib.pyplot as plt
df = pd.read_csv('talegaon_final_clean.csv')
df['Date'] = pd.to_datetime(df['Date'])
bulk_index = df[df['Order_Total'] == 37502.1].index
df = df.drop(bulk_index[1:])
print(df.shape)

(113474, 14)


In [5]:
customer_activity = df.groupby('Customer_ID').agg(first_order = ('Date','min'),last_order = ('Date','max'),total_order = ('Date','count'), total_revenue = ('Order_Total','sum')).reset_index()

In [10]:
latest_date = df['Date'].max()
customer_activity['day_since_last_order'] = (latest_date - customer_activity['last_order']).dt.days

In [14]:
def segment_customer(days):
    if days <= 60:
        return 'Active'
    elif days <= 180:
        return 'At risk'
    else:
        return 'Churned'
customer_activity['segment'] = customer_activity['day_since_last_order'].apply(segment_customer)
print(customer_activity.shape)
print(customer_activity['segment'].value_counts())

(4460, 7)
segment
Churned    2443
Active     1399
At risk     618
Name: count, dtype: int64


In [24]:
rfm = customer_activity[['Customer_ID', 'day_since_last_order', 'total_order', 'total_revenue']]
print(rfm.head())

      Customer_ID  day_since_last_order  total_order  total_revenue
0       ADJ_APR26                    42            2        60000.0
1  MOB_0172167995                   994            1          600.0
2  MOB_1450401213                   441           26        23520.0
3  MOB_1451994862                    88            4          560.0
4  MOB_1504564834                   830            2        15000.0


In [25]:
rfm.columns = ['Customer_ID','Recency','Frequency','Monetary']
print(rfm.head())

      Customer_ID  Recency  Frequency  Monetary
0       ADJ_APR26       42          2   60000.0
1  MOB_0172167995      994          1     600.0
2  MOB_1450401213      441         26   23520.0
3  MOB_1451994862       88          4     560.0
4  MOB_1504564834      830          2   15000.0


In [40]:
rfm = customer_activity[['Customer_ID', 'day_since_last_order', 'total_order', 'total_revenue']].copy()
rfm.columns = ['Customer_ID', 'Recency', 'Frequency', 'Monetary']

rfm['R_Score'] = pd.cut(rfm['Recency'], bins=3, labels=[3,2,1])
rfm['F_Score'] = pd.cut(rfm['Frequency'], bins=3, labels=[1,2,3])
rfm['M_Score'] = pd.cut(rfm['Monetary'], bins=3, labels=[1,2,3])

rfm['RFM_Score'] = rfm['R_Score'].astype(int) + rfm['F_Score'].astype(int) + rfm['M_Score'].astype(int)

print(rfm.head(10))

      Customer_ID  Recency  Frequency  Monetary R_Score F_Score M_Score  \
0       ADJ_APR26       42          2   60000.0       3       1       1   
1  MOB_0172167995      994          1     600.0       2       1       1   
2  MOB_1450401213      441         26   23520.0       3       1       1   
3  MOB_1451994862       88          4     560.0       3       1       1   
4  MOB_1504564834      830          2   15000.0       2       1       1   
5  MOB_1507461567      494          7   12950.0       3       1       1   
6  MOB_1509002374       11          2     280.0       3       1       1   
7  MOB_1544069900      162          1    1200.0       3       1       1   
8  MOB_1589102341      228          9    3630.0       3       1       1   
9  MOB_2223058131     1837          4     880.0       1       1       1   

   RFM_Score  
0          5  
1          4  
2          5  
3          5  
4          4  
5          5  
6          5  
7          5  
8          5  
9          3  


In [50]:
rfm = customer_activity[['Customer_ID', 'day_since_last_order', 'total_order', 'total_revenue']].copy()
rfm.columns = ['Customer_ID', 'Recency', 'Frequency', 'Monetary']

rfm['R_Score'] = pd.qcut(rfm['Recency'].rank(method='first'),q=3, labels=[3,2,1])
rfm['F_Score'] = pd.qcut(rfm['Frequency'].rank(method='first'),q=3, labels=[1,2,3])
rfm['M_Score'] = pd.qcut(rfm['Monetary'].rank(method='first'),q=3 ,labels=[1,2,3])

rfm['RFM_Score'] = rfm['R_Score'].astype(int) + rfm['F_Score'].astype(int) + rfm['M_Score'].astype(int)

print(rfm[['Customer_ID','R_Score','F_Score','M_Score','RFM_Score']].head(10))
print("\nScore distribution:")
print(rfm['RFM_Score'].value_counts().sort_index())


      Customer_ID R_Score F_Score M_Score  RFM_Score
0       ADJ_APR26       3       1       3          7
1  MOB_0172167995       1       1       1          3
2  MOB_1450401213       2       3       3          8
3  MOB_1451994862       2       2       1          5
4  MOB_1504564834       1       1       3          5
5  MOB_1507461567       2       2       3          7
6  MOB_1509002374       3       1       1          5
7  MOB_1544069900       2       1       1          4
8  MOB_1589102341       2       2       2          6
9  MOB_2223058131       1       2       1          4

Score distribution:
RFM_Score
3    579
4    641
5    811
6    629
7    532
8    506
9    762
Name: count, dtype: int64


In [60]:
def rfm_segment(score):
    if score >= 8:
         return('Champion')
    elif score >= 7:
         return('Loyal')
    elif score >= 5:
         return('Potential')
    else :
        return('Lost')

In [61]:
rfm['RFM_Segment'] = rfm['RFM_Score'].apply(rfm_segment)
print(rfm['RFM_Segment'].value_counts())

RFM_Segment
Potential    1440
Champion     1268
Lost         1220
Loyal         532
Name: count, dtype: int64


In [66]:
segment_counts = rfm['RFM_Segment'].value_counts()
colors = ['#2ecc71', '#3498db', '#f39c12', '#e74c3c']
plt.figure(figsize=(8,6))
plt.bar(segment_counts.index,segment_count.values,color=colors)
plt.title('Customer Segments by RFM Score')
plt.xlabel('Segment')
plt.ylabel('Number of customer')
plt.tight_layout()
plt.show()

NameError: name 'segment_count' is not defined

<Figure size 800x600 with 0 Axes>